# 02 · AVA-VLA 전체 미세조정 → 검증 후 T1 선택 → 순차 캐시
01을 완료한 뒤 A100 런타임에서 실행하십시오. LoRA를 사용하지 않으며 backbone/head/proprio/attention generator를 학습 대상으로 둡니다.
A100 40 GB에서 FFT는 OOM이 날 수 있습니다. 먼저 1 step을 실행하고 실제 peak 메모리를 확인하십시오.
**현재 FFT는 temporal zero-context입니다.** 이후 캐시는 이전 query의 layer -2 action hidden을 전달하지만,
시간 문맥을 사용한 FFT/TBPTT를 구현하거나 원본 AVA 학습을 재현한 것은 아닙니다. 본 실험 채택 전 검토 항목입니다.
체크포인트/optimizer 저장은 수십 GB가 될 수 있습니다. `COMPLETE`와 training state가 있는 저장만 재개합니다.

In [ ]:
import os, time
SESSION_STARTED_UNIX = globals().get('SESSION_STARTED_UNIX', time.time())
SESSION_DEADLINE_UNIX = SESSION_STARTED_UNIX + 8 * 3600
os.environ['VLADE_DEADLINE_UNIX'] = str(SESSION_DEADLINE_UNIX)
print('A100 예산: 설치·파일럿·학습·후속 작업을 포함한 최대 8시간')

In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib, json, os, shutil, subprocess, sys
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/colab/VLADE')  # 세 노트북에서 동일한 저장 폴더를 지정하십시오.
ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = ROOT  # src/scripts/configs/experiments/third_party가 바로 들어 있는 실제 프로젝트 루트
EXPECTED_SOURCE_MANIFEST_SHA256 = 'd144261f04ca80b7f29028b32375a6b7cee43e3868c55a55221523801010de4a'
def stage_source(project_root: Path, scratch_root: Path, expected_manifest_sha256: str) -> Path:
    """Verify Drive source files and stage them without copying data or weights."""
    project_root = project_root.resolve()
    manifest_relative = Path("experiments/manifests/colab_source.json")
    manifest_path = project_root / manifest_relative
    if not manifest_path.is_file():
        raise FileNotFoundError("Drive의 VLADE 폴더에 src/scripts/configs/experiments/third_party 일반 파일 구조를 먼저 복사하십시오: " + str(manifest_path))
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != expected_manifest_sha256:
        raise ValueError("노트북과 colab_source.json 버전이 다릅니다. 같은 버전의 일반 프로젝트 파일을 사용하십시오.")
    manifest = json.loads(manifest_bytes)
    code = scratch_root / ("vlade_colab_" + expected_manifest_sha256[:12])
    code = code.resolve()
    if code.is_relative_to(project_root) or project_root.is_relative_to(code):
        raise ValueError("실행 소스 임시 폴더와 Drive 프로젝트 폴더는 서로 겹칠 수 없습니다.")
    for relative, expected in manifest["files_sha256"].items():
        path = Path(relative)
        if path.is_absolute() or ".." in path.parts or "\\" in relative or ":" in relative:
            raise ValueError("잘못된 소스 상대 경로: " + relative)
        source = project_root / path
        target = code / path
        if not source.resolve().is_relative_to(project_root) or not target.resolve().is_relative_to(code):
            raise ValueError("프로젝트 밖의 소스 경로: " + relative)
        if not source.is_file() or hashlib.sha256(source.read_bytes()).hexdigest() != expected:
            raise ValueError("Drive 소스가 없거나 변경되었습니다: " + relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            shutil.copyfile(source, target)
        if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError("복사된 소스 무결성 불일치: " + relative)
    # Publish the manifest only after every listed source file has been verified.
    staged_manifest = code / manifest_relative
    staged_manifest.parent.mkdir(parents=True, exist_ok=True)
    temporary_manifest = staged_manifest.with_suffix(".json.tmp")
    temporary_manifest.write_bytes(manifest_bytes)
    temporary_manifest.replace(staged_manifest)
    return code

CODE = stage_source(PROJECT_ROOT, Path('/content'), EXPECTED_SOURCE_MANIFEST_SHA256)
for module_name, module in list(sys.modules.items()):
    if module_name == 'vlade' or module_name.startswith('vlade.'):
        filename = getattr(module, '__file__', None)
        if filename and not Path(filename).resolve().is_relative_to(CODE.resolve()):
            raise RuntimeError('다른 버전의 VLADE가 메모리에 있습니다. Colab 런타임을 다시 시작하십시오.')
sys.path.insert(0, str(CODE / 'src'))
print('Drive:', ROOT, '\n실행 소스:', CODE)
ENV = os.environ.copy()
ENV.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', TF_CPP_MIN_LOG_LEVEL='2',
           HF_HOME=str(ROOT / 'hf_cache'), HF_TOKEN_PATH='/content/vlade_hf_token')
def run(args, env=None):
    import signal, threading, time
    deadline = float((env or ENV).get('VLADE_DEADLINE_UNIX', '0'))
    if deadline and deadline - time.time() <= 60:
        raise RuntimeError('8시간 예산이 끝났습니다. 새 작업을 시작하지 않습니다.')
    process = subprocess.Popen([str(x) for x in args], cwd=CODE, env=env or ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timers = []
    if deadline:
        for delay, callback in [(max(0, deadline-time.time()-60), lambda: process.send_signal(signal.SIGINT)),
                                (max(0, deadline-time.time()), process.kill)]:
            timer = threading.Timer(delay, callback)
            timer.daemon = True
            timer.start()
            timers.append(timer)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('실행에 실패했거나 시간 예산이 끝났습니다. 위 결과를 확인하십시오.')
    finally:
        for timer in timers:
            timer.cancel()
UV = shutil.which('uv')
if UV is None:
    tool_env = Path('/content/vlade_uv_tool')
    tool_python = tool_env / 'bin/python'
    if not tool_python.is_file():
        run([sys.executable, '-m', 'venv', tool_env])
    run([tool_python, '-m', 'pip', 'install', 'uv'])
    UV = str(tool_env / 'bin/uv')
assert UV
def environment(name, version, packages):
    directory = Path('/content') / name
    python = directory / 'bin/python'
    if not python.is_file():
        run([UV, 'venv', '--python', version, directory])
    run([UV, 'pip', 'install', '--python', python, *packages])
    return python
def completed_checkpoints(directory):
    return sorted(path for path in directory.glob('step-*')
                  if (path / 'COMPLETE').is_file() and (path / 'trainer_state.pt').is_file())

# HF inputs are cached on the runtime SSD; Drive retains code and experiment records.
ASSET_CACHE = Path('/content/vlade_hf_assets')
ASSET_WORK_ROOT = Path('/content/vlade_asset_work')
RUNS_ROOT = Path('/content/vlade_runs')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
ENV.update(VLADE_ASSET_CACHE=str(ASSET_CACHE),
           VLADE_ASSET_REGISTRY=str(CODE/'configs/storage/hf_assets.lock.json'),
           HF_HOME='/content/vlade_hf_home', HF_TOKEN_PATH='/content/vlade_hf_token')
os.environ.update({key: ENV[key] for key in ('VLADE_ASSET_CACHE','VLADE_ASSET_REGISTRY')})
_HF_REGISTRY_READY = False

def hf_input(key, python=None, *, fetch=True):
    global _HF_REGISTRY_READY
    from vlade.storage.assets import asset_record, cache_path
    if fetch and not _HF_REGISTRY_READY:
        if python is None:
            raise ValueError('Select the prepared Python environment before fetching HF assets')
        registry = ASSET_CACHE / 'registry.json'
        run([python, 'scripts/data/hf_assets.py', 'refresh-registry', '--output', registry], env=ENV)
        ENV['VLADE_ASSET_REGISTRY'] = str(registry)
        os.environ['VLADE_ASSET_REGISTRY'] = str(registry)
        _HF_REGISTRY_READY = True
    record = asset_record(key, Path(ENV['VLADE_ASSET_REGISTRY']))
    target = cache_path(key, record, ASSET_CACHE)
    if fetch:
        if python is None:
            raise ValueError('Select the prepared CPU/model Python environment before fetching HF assets')
        run([python, 'scripts/data/hf_assets.py', 'fetch', key,
             '--registry', ENV['VLADE_ASSET_REGISTRY'], '--cache', ASSET_CACHE], env=ENV)
    return target

def hf_path(value, python=None, *, fetch=True):
    value = str(value)
    return hf_input(value[len('hfasset:'):].strip('/\\'), python, fetch=fetch) if value.startswith('hfasset:') else Path(value)

In [ ]:
# HF 설정 후 기존 학습 셀을 실행하면 checkpoint 저장 시 자동 전송합니다.
# 저장소는 HF 웹에서 Private로 먼저 만들고 해당 저장소 쓰기 토큰을 Secrets에 등록하십시오.
if 'ENV' not in globals():
    ENV = os.environ.copy()
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
ENV['HF_TOKEN'] = os.environ['HF_TOKEN']
import datetime, uuid
ENABLE_HUB_UPLOAD = globals().get('ENABLE_HUB_UPLOAD', True)
HF_REPO_ID = 'VLADE-konkuk/VLADE-checkpoints'  # 조직 저장소; 이전 런타임 값도 갱신합니다.
HF_RUN_GROUP = globals().get('HF_RUN_GROUP', datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])  # 재개 시 기존 group 지정
if ENABLE_HUB_UPLOAD:
    if not HF_RUN_GROUP:
        raise ValueError('HF_RUN_GROUP을 지정하십시오. pilot/main은 출력 폴더명으로 구분합니다.')
    os.environ['VLADE_HUB_REPO'] = HF_REPO_ID
    os.environ['VLADE_HUB_RUN_ID'] = HF_RUN_GROUP
    os.environ['HF_TOKEN_PATH'] = '/content/vlade_hf_token'
    # 모델 캐시는 기존 위치를 유지하고 토큰 파일은 런타임 로컬에만 둡니다.
    ENV.update(HF_TOKEN=os.environ['HF_TOKEN'], VLADE_HUB_REPO=HF_REPO_ID,
               VLADE_HUB_RUN_ID=HF_RUN_GROUP, HF_TOKEN_PATH='/content/vlade_hf_token')
    print('Private Hub 업로드 설정:', HF_REPO_ID, HF_RUN_GROUP)
else:
    for key in ('VLADE_HUB_REPO', 'VLADE_HUB_RUN_ID'):
        ENV.pop(key, None)
        os.environ.pop(key, None)
    print('Hub 업로드 꺼짐: HF_RUN_GROUP 설정 후 ENABLE_HUB_UPLOAD=True로 이 셀을 다시 실행하십시오.')

In [ ]:
import threading
from google.colab import runtime
POLICY = json.loads((CODE / 'configs/experiments/ava_fft_a100_8h.json').read_text())
RUN_NAME = 'ava-fft-a100-8h-seed42'
CONFIGS = ROOT / 'run-configs'
CONFIGS.mkdir(exist_ok=True)
SESSION_FILE = CONFIGS / (RUN_NAME + '-session.json')
if SESSION_FILE.is_file():
    session = json.loads(SESSION_FILE.read_text())
    SESSION_STARTED_UNIX = session['started_unix']
    SESSION_DEADLINE_UNIX = session['deadline_unix']
else:
    SESSION_FILE.write_text(json.dumps(dict(started_unix=SESSION_STARTED_UNIX,
        deadline_unix=SESSION_DEADLINE_UNIX, total_hours=POLICY['total_hours']), indent=2))
ENV['VLADE_DEADLINE_UNIX'] = str(SESSION_DEADLINE_UNIX)
os.environ['VLADE_DEADLINE_UNIX'] = str(SESSION_DEADLINE_UNIX)
if SESSION_DEADLINE_UNIX <= time.time():
    runtime.unassign()
    raise RuntimeError('이 실행의 8시간 예산이 종료되었습니다.')
if globals().get('_budget_release_timer'):
    _budget_release_timer.cancel()
_budget_release_timer = threading.Timer(SESSION_DEADLINE_UNIX-time.time(), runtime.unassign)
_budget_release_timer.daemon = True
_budget_release_timer.start()
print('남은 예산(분):', round((SESSION_DEADLINE_UNIX-time.time())/60, 1))
print('8시간 마감 시 Colab 런타임 자동 해제를 요청합니다. 대기 시간도 포함됩니다.')

In [ ]:
AVA_ENV_READY = False
AVA_PY = None
import tempfile, tomllib, zipfile
AVA_VENV = Path('/content') / 'vlade_ava_fft_py311'
AVA_PATCH_REL = 'configs/models/teacher/transformers-avavla-a03eee5-checkpoint-extra-attn.patch'
AVA_PATCH_SHA = '9f0e46c9e93d62a92af950aeee712191cfcf314c1e636a9608fddcb57ff6087d'
AVA_TRANSFORMERS_REV = 'a03eee5da1794c45f5a911b60a9a4545e6e0ed1f'
AVA_BASE_SHA = '62f22856fe7b8459d6d61d4676b0075b65724979f84859ac475dbe36c10a3d80'
AVA_INSTALLED_SHA = '0a9c25300f3358f87af1719199608af3c27567dba0425690327fccd44cfb516b'
AVA_VERIFY_SOURCE = "import ast, hashlib, importlib, json, sys\nfrom pathlib import Path\nexpected_sha, expected_prefix = sys.argv[1:3]\nprefix = Path(sys.prefix).resolve()\nif sys.prefix == sys.base_prefix or prefix != Path(expected_prefix).resolve():\n    raise RuntimeError('AVA verification requires the dedicated virtual environment')\nmodule = importlib.import_module('transformers.models.llama.modeling_llama')\nsource = Path(module.__file__).resolve()\nif not source.is_relative_to(prefix) or source.name != 'modeling_llama.py':\n    raise RuntimeError('Transformers was imported outside the AVA environment')\npayload = source.read_bytes()\nif hashlib.sha256(payload).hexdigest() != expected_sha:\n    raise RuntimeError('Installed AVA Transformers source SHA mismatch')\ntree = ast.parse(payload)\nmodel = next(node for node in tree.body if isinstance(node, ast.ClassDef) and node.name == 'LlamaModel')\nforward = next(node for node in model.body if isinstance(node, ast.FunctionDef) and node.name == 'forward')\ncalls = [node for node in ast.walk(forward) if isinstance(node, ast.Call)\n         and isinstance(node.func, ast.Attribute) and node.func.attr == '_gradient_checkpointing_func']\nexpected = 'extra_attn_weights if extra_layer_ids and layer_idx in extra_layer_ids else None'\nif len(calls) != 1 or len(calls[0].args) != 9 or ast.unparse(calls[0].args[-1]) != expected:\n    raise RuntimeError('Installed checkpoint attention argument was not preserved')\nprint(json.dumps({'source': str(source), 'sha256': expected_sha, 'venv': str(prefix)}))"
_VLADE_AVA_BASE_RUN = getattr(run, '_vlade_base_run', run)

def verify_ava_transformers(python):
    _VLADE_AVA_BASE_RUN([python, '-c', AVA_VERIFY_SOURCE, AVA_INSTALLED_SHA, AVA_VENV])

def require_ava_environment():
    if not AVA_ENV_READY or AVA_PY is None:
        raise RuntimeError('AVA patched environment is not verified; rerun the installation cell')
    verify_ava_transformers(AVA_PY)

def run(args, env=None):
    command = [str(value) for value in args]
    if command and (command[0] == str(AVA_VENV / 'bin/python')
                    or any(Path(value).name == 'run_fft.py' for value in command)):
        require_ava_environment()
    return _VLADE_AVA_BASE_RUN(args, env=env)

run._vlade_base_run = _VLADE_AVA_BASE_RUN
patch = CODE / AVA_PATCH_REL
source_manifest = json.loads((CODE / 'experiments/manifests/colab_source.json').read_text())
if source_manifest['files_sha256'].get(AVA_PATCH_REL) != AVA_PATCH_SHA:
    raise RuntimeError('Staged source manifest does not pin the approved AVA patch')
if not patch.is_file() or hashlib.sha256(patch.read_bytes()).hexdigest() != AVA_PATCH_SHA:
    raise RuntimeError('Staged AVA patch is missing or has the wrong SHA')

build_root = Path(tempfile.mkdtemp(prefix='vlade-ava-transformers-', dir=AVA_VENV.parent))
source = build_root / 'source'
source.mkdir()
build_env = ENV.copy()
build_env.update(UV_CACHE_DIR=str(build_root / 'uv-cache'), TMPDIR=str(build_root / 'tmp'))
(build_root / 'tmp').mkdir()
fork = 'https://github.com/LiAuto-DSR/transformers-avavla.git'
_VLADE_AVA_BASE_RUN(['git', '-C', source, 'init'], env=build_env)
_VLADE_AVA_BASE_RUN(['git', '-C', source, 'fetch', '--depth', '1', fork, AVA_TRANSFORMERS_REV], env=build_env)
_VLADE_AVA_BASE_RUN(['git', '-C', source, 'checkout', '--detach', 'FETCH_HEAD'], env=build_env)
revision = subprocess.check_output(['git', '-C', str(source), 'rev-parse', 'HEAD'], text=True).strip()
if revision != AVA_TRANSFORMERS_REV:
    raise RuntimeError('AVA Transformers checkout revision mismatch')
llama = source / 'src/transformers/models/llama/modeling_llama.py'
if hashlib.sha256(llama.read_bytes()).hexdigest() != AVA_BASE_SHA:
    raise RuntimeError('Unpatched AVA Transformers source SHA mismatch')
_VLADE_AVA_BASE_RUN(['git', '-C', source, 'apply', '--check', patch], env=build_env)
_VLADE_AVA_BASE_RUN(['git', '-C', source, 'apply', patch], env=build_env)
if hashlib.sha256(llama.read_bytes()).hexdigest() != AVA_INSTALLED_SHA:
    raise RuntimeError('Patched AVA Transformers source SHA mismatch')
wheels = build_root / 'wheels'
wheels.mkdir()
_VLADE_AVA_BASE_RUN([UV, 'build', '--wheel', '--python', '3.11', '--out-dir', wheels, source], env=build_env)
wheel_files = list(wheels.glob('transformers-*.whl'))
if len(wheel_files) != 1:
    raise RuntimeError('Expected exactly one patched Transformers wheel')
wheel = wheel_files[0]
with zipfile.ZipFile(wheel) as archive:
    wheel_source = archive.read('transformers/models/llama/modeling_llama.py')
if hashlib.sha256(wheel_source).hexdigest() != AVA_INSTALLED_SHA:
    raise RuntimeError('Built Transformers wheel does not contain the approved patch')

upstream = CODE / 'third_party/ava_vla'
dependencies = tomllib.loads((upstream / 'pyproject.toml').read_text())['project']['dependencies']
dependencies = [dep for dep in dependencies if not dep.startswith(('transformers @', 'dlimp @', 'huggingface_hub', 'accelerate'))]
dependencies += [
    str(wheel),
    'dlimp @ git+https://github.com/moojink/dlimp_openvla.git@040105d256bd28866cc6620621a3d5f7b6b91b46',
    'numpy==1.26.4', 'tensorflow-metadata==1.15.0', 'bitsandbytes==0.43.3',
    'huggingface_hub==0.36.2', 'accelerate==1.15.0']
candidate_python = environment('vlade_ava_fft_py311', '3.11', dependencies)
if str(candidate_python) != str(AVA_VENV / 'bin/python'):
    raise RuntimeError('Refusing to install outside the dedicated AVA environment')
_VLADE_AVA_BASE_RUN([UV, 'pip', 'install', '--python', candidate_python,
                    '--no-deps', '--force-reinstall', wheel], env=build_env)
_VLADE_AVA_BASE_RUN([UV, 'pip', 'install', '--python', candidate_python, '--no-deps', '-e', upstream])
verify_ava_transformers(candidate_python)
_VLADE_AVA_BASE_RUN([candidate_python, '-c', "import torch,transformers,tensorflow_datasets; assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(); print(torch.__version__,transformers.__version__,torch.cuda.get_device_name(0),torch.cuda.get_device_properties(0).total_memory/2**30)"])
freeze = subprocess.check_output([str(UV), 'pip', 'freeze', '--python', str(candidate_python)], text=True)
(ROOT / 'ava_environment_freeze.txt').write_text(freeze)
(AVA_VENV / 'vlade_transformers_patch.json').write_text(json.dumps({
    'base_commit': AVA_TRANSFORMERS_REV, 'patch_sha256': AVA_PATCH_SHA,
    'installed_source_sha256': AVA_INSTALLED_SHA,
    'wheel_sha256': hashlib.sha256(wheel.read_bytes()).hexdigest()}, indent=2))
AVA_PY = candidate_python
AVA_ENV_READY = True
print('AVA managed Transformers patch verified; FFT environment ready')

In [ ]:
require_ava_environment()
PREPARED = hf_input('prepared_d_seed42', AVA_PY)
from vlade.data.episode_data import EpisodeData, query_count
from vlade.training.run_control import require_reports, planned_checkpoint_space, pilot_context
from vlade.training.compute_budget import pilot_profile, plan_fft
RUN_MAIN_FFT = False  # 측정된 예산과 재로딩 결과 확인 후 True
OUTPUT = RUNS_ROOT / 'ava_fft' / RUN_NAME
TRAIN_QUERIES = query_count(EpisodeData(PREPARED))
config = json.loads((CODE / 'configs/experiments/libero_plus_fft.json').read_text())
config['fft_defaults'].update(data_root=str(PREPARED),
    prepared_data_root=str(PREPARED), image_aug=False, batch_size=POLICY['batch_size'],
    grad_accumulation_steps=POLICY['accumulation_steps'], max_steps=POLICY['pilot_steps'],
    save_steps=POLICY['pilot_steps'], save_optimizer_state=True,
    learning_rate=POLICY['learning_rate'], lr_schedule=POLICY['lr_schedule'], warmup_steps=0,
    max_grad_norm=POLICY['max_grad_norm'], eval_steps=POLICY['pilot_steps'],
    max_eval_batches=POLICY['validation_batches'], debug_records=0)
config['fft_runs']['ava_vla'].update(
    checkpoint=str(hf_input('ava_t0', AVA_PY)), output_dir=str(OUTPUT))
MAIN_CONFIG = CONFIGS / (RUN_NAME + '.json')
print('train query:', TRAIN_QUERIES, ' / 본 학습 max_steps: 파일럿 실측 후 계산')

## 동일 accumulation의 4-step 파일럿 → 저장·재로딩 검사
첫 optimizer step은 속도 산정에서 제외하고 뒤의 3 step, validation 및 Drive 저장 시간을 측정합니다.
파일럿은 본 학습과 같은 batch/accumulation을 사용합니다. 파일럿 가중치는 본 학습 초기값으로 쓰지 않습니다.

In [ ]:
require_ava_environment()
smoke = json.loads(json.dumps(config))
SMOKE_OUTPUT = RUNS_ROOT / 'ava_fft' / (RUN_NAME + '-pilot4')
SMOKE_STEP = SMOKE_OUTPUT / f"step-{POLICY['pilot_steps']:07d}"
smoke['fft_runs']['ava_vla']['output_dir'] = str(SMOKE_OUTPUT)
SMOKE_CONFIG = CONFIGS / (RUN_NAME + '-pilot4.json')
if SMOKE_CONFIG.exists() and json.loads(SMOKE_CONFIG.read_text()) != smoke:
    raise ValueError('기존 파일럿과 설정이 다릅니다. 같은 경로를 덮어쓰지 않습니다.')
SMOKE_CONFIG.write_text(json.dumps(smoke, indent=2))
if not (SMOKE_STEP / 'COMPLETE').is_file():
    run([AVA_PY, 'scripts/train/run_fft.py', '--model', 'ava_vla', '--config', SMOKE_CONFIG])
run([AVA_PY, 'scripts/train/check_fft_checkpoint.py', '--checkpoint', SMOKE_STEP,
     '--data', PREPARED, '--config', SMOKE_CONFIG, '--report', SMOKE_OUTPUT / 'reload_report.json'])
PROFILE = pilot_profile(SMOKE_OUTPUT, expected_steps=POLICY['pilot_steps'])
current_hardware = json.loads(subprocess.check_output([str(AVA_PY), '-c',
    "import json,torch; print(json.dumps(dict(gpu=torch.cuda.get_device_name(0), vram_bytes=torch.cuda.get_device_properties(0).total_memory)))"], text=True))
if ('A100' not in current_hardware['gpu'] or
        any(PROFILE[key] != current_hardware[key] for key in ('gpu', 'vram_bytes'))):
    raise RuntimeError('현재 A100과 timing pilot의 GPU/메모리가 다릅니다. 기존 측정값을 재사용하지 않습니다.')
print('실측 시간(초):', json.dumps(PROFILE, indent=2))

In [ ]:
require_reports([SMOKE_OUTPUT / 'reload_report.json'],
    [pilot_context(SMOKE_CONFIG, SMOKE_STEP, PREPARED, 'fft')])
if MAIN_CONFIG.is_file():
    existing = json.loads(MAIN_CONFIG.read_text())
    if existing.get('training', {}).get('budget_policy') != POLICY:
        raise ValueError('기존 본 학습의 예산 정책이 다릅니다. 새 실행 이름이 필요합니다.')
    if any(existing['fft_defaults'].get(k) != v for k, v in config['fft_defaults'].items()
           if k not in {'max_steps', 'save_steps', 'eval_steps', 'warmup_steps'}):
        raise ValueError('기존 본 학습과 입력/학습 설정이 다릅니다.')
    config = existing  # 재개 시 max_steps와 LR 스케줄을 재계산하지 않습니다.
    BUDGET = config['training']['colab_budget']
else:
    BUDGET = plan_fft(TRAIN_QUERIES, PROFILE, SESSION_DEADLINE_UNIX-time.time(),
        accumulation=POLICY['accumulation_steps'], max_epochs=POLICY['max_epochs'],
        safety_factor=POLICY['safety_factor'],
        downstream_reserve_seconds=POLICY['downstream_reserve_seconds'],
        shutdown_reserve_seconds=POLICY['shutdown_reserve_seconds'])
    config['fft_defaults'].update(max_steps=BUDGET['optimizer_steps'],
        save_steps=BUDGET['save_steps'], eval_steps=BUDGET['eval_steps'],
        warmup_steps=BUDGET['warmup_steps'])
    config['training'].update(colab_budget=BUDGET, budget_policy=POLICY)
    MAIN_CONFIG.write_text(json.dumps(config, indent=2))
print('본 학습 예산:', json.dumps(BUDGET, indent=2))
if POLICY['compute_units_per_hour'] is not None:
    print('8시간 최대 예상 CU:', POLICY['total_hours'] * POLICY['compute_units_per_hour'])

## 8시간 예산 내 본 학습
최대 1 epoch와 실측 가능한 step 중 작은 값을 사용합니다. 설치·파일럿·재로딩은 이미 경과한 시간으로 차감됩니다.
이후 validation/T1/cache 작업용 1시간과 종료용 최소 5분을 확보하고, 나머지 속도 추정에는 30% 여유를 둡니다.
저장·validation은 중간과 마지막 약 2회로 줄입니다. 실제 저장이 느려지면 optimizer 경계에서 저장 후 예산 종료로 기록합니다.
이 설정은 논문 재현/수렴 완료를 뜻하지 않습니다. FFT의 LR은 2e-5, warm-up 10%, cosine이며 temporal zero-context입니다.
RUN_MAIN_FFT를 켜야 학습합니다. 8시간 후 런타임 해제 요청이 실패하면 Colab에서 직접 연결 해제·삭제하십시오.

In [ ]:
require_ava_environment()
checkpoints = completed_checkpoints(OUTPUT)
latest = checkpoints[-1] if checkpoints else None
if not RUN_MAIN_FFT:
    print('본 학습 대기: 측정 예산과 재로딩 결과 확인 후 RUN_MAIN_FFT=True로 실행하십시오.')
elif latest and int(latest.name.split('-')[1]) >= config['fft_defaults']['max_steps']:
    print('목표 step 완료:', latest)
else:
    require_reports([SMOKE_OUTPUT / 'reload_report.json'],
        [pilot_context(SMOKE_CONFIG, SMOKE_STEP, PREPARED, 'fft')])
    print('저장 공간 계획:', planned_checkpoint_space(RUNS_ROOT, SMOKE_STEP,
        config['fft_defaults']['max_steps'], config['fft_defaults']['save_steps'],
        int(latest.name.split('-')[1]) if latest else 0))
    train_env = ENV.copy()
    train_env['VLADE_TRAIN_DEADLINE_UNIX'] = str(SESSION_DEADLINE_UNIX-POLICY['downstream_reserve_seconds'])
    train_env['VLADE_SAVE_RESERVE_SECONDS'] = str(BUDGET['save_reserve_seconds'])
    command = [AVA_PY, 'scripts/train/run_fft.py', '--model', 'ava_vla', '--config', MAIN_CONFIG]
    if latest:
        command += ['--resume', latest]
    run(command, env=train_env)

## Validation 후 T1 선택
마지막 step을 자동으로 T1으로 부르지 않습니다. 별도 validation/rollout 보고서를 확인한 뒤 아래 세 값을 지정하십시오.
`validation_report`는 JSON이며 report_type=teacher_validation_comparison, selection_split=validation,
used_final_test=false, 실제 evaluation_protocol_sha256, prepared manifest의 data_split_sha256,
candidates 배열을 포함해야 합니다. 각 후보에는 checkpoint/role(T0 또는 T1),
validation_success_rate/validation_episodes, original_libero_success_rate/original_libero_episodes가 필요합니다.
실제 측정값과 선택 T1이 없으면 거부합니다. 한 번의 재로딩 보고서는 선정 근거가 아닙니다.
선택 파일에 전체 bundle SHA256·통계/split SHA256·보고서 SHA256을 기록합니다.

In [ ]:
T1_CHECKPOINT = ''  # 실제 validation으로 선택한 완료 checkpoint의 Drive 경로
VALIDATION_REPORT = ''  # 실제 validation/rollout 보고서의 Drive 경로
SELECTION_REASON = ''  # 선택 지표·비교 checkpoint·이유
SELECTION = ROOT / 'selections/T1.json'
if T1_CHECKPOINT and VALIDATION_REPORT and SELECTION_REASON:
    if SELECTION.is_file():
        previous = json.loads(SELECTION.read_text())
        if (Path(previous['checkpoint']).resolve() != Path(T1_CHECKPOINT).resolve() or
            Path(previous['validation_report']).resolve() != Path(VALIDATION_REPORT).resolve() or
            previous['selection_reason'] != SELECTION_REASON or
            hashlib.sha256(Path(VALIDATION_REPORT).read_bytes()).hexdigest() != previous['validation_report_sha256']):
            raise ValueError('기존 T1 선택과 새 입력이 다릅니다. SELECTION에 새 파일 경로를 지정하십시오.')
    else:
        run([AVA_PY, 'scripts/data/select_teacher.py', '--checkpoint', T1_CHECKPOINT,
             '--validation-report', VALIDATION_REPORT, '--reason', SELECTION_REASON,
             '--data', PREPARED, '--output', SELECTION])
else:
    print('선행 대기: 실제 validation·T1 선정 후 cache를 생성하십시오.')

## 고정 시퀀스 cache: 2-episode pilot → 전체 train
T1은 frozen·비양자화, query 간격 8, episode별 reset과 이전 hidden 전달을 유지합니다.
재로딩 무결성·샘플 수와 고르게 떨어진 최대 3개 episode의 전체 query 재실행을 검사하고 완료된 episode만 재사용합니다.
기본은 action cache입니다. `SAVE_HIDDEN=True`이면 `[56,D]` layer -2 feature도 기록합니다.
큰 hidden JSON은 전체 데이터에서 용량이 매우 커질 수 있으므로 pilot 실측으로 예산을 먼저 확인하십시오.
`RUN_FULL_CACHE`는 pilot 결과를 검토한 후 켜십시오. action-only cache를 representation KD용으로 해석하지 않습니다.

In [ ]:
SAVE_HIDDEN = False
RUN_FULL_CACHE = False
if SELECTION.is_file():
    suffix = 'hidden' if SAVE_HIDDEN else 'action'
    PILOT = ASSET_WORK_ASSET_WORK_ASSET_WORK_ASSET_WORK_ASSET_WORK_ASSET_WORK_ROOT / 'data/teacher_cache' / ('T1-pilot-' + suffix)
    command = [AVA_PY, 'scripts/data/generate_fixed_cache.py', '--selection', SELECTION,
               '--data', PREPARED, '--output', PILOT, '--max-episodes', '2']
    run(command + (['--hidden'] if SAVE_HIDDEN else []))
    pilot = json.loads((PILOT / 'cache_manifest.json').read_text())
    total_queries = sum(item['queries'] for item in json.loads((PREPARED / 'manifest.json').read_text())['episodes'] if item['split'] == 'train')
    estimate = pilot['bytes'] * total_queries / pilot['queries']
    print('전체 cache 예상 GiB (pilot 비례 추정):', estimate / 2**30)
    estimated_seconds = pilot['invocation_seconds'] * total_queries / pilot['queries'] * POLICY['safety_factor']
    print('전체 cache 예상 시간(분, pilot 비례 추정):', estimated_seconds / 60)
    if RUN_FULL_CACHE:
        if estimated_seconds + POLICY['shutdown_reserve_seconds'] > SESSION_DEADLINE_UNIX-time.time():
            raise RuntimeError('남은 8시간 예산으로 전체 cache를 생성하기 어렵습니다. pilot은 보존합니다.')
        if shutil.disk_usage(RUNS_ROOT).free < estimate * 1.2:
            raise OSError('pilot 기반 cache 예상 용량과 여유 공간이 부족합니다.')
        command = [AVA_PY, 'scripts/data/generate_fixed_cache.py', '--selection', SELECTION,
                   '--data', PREPARED, '--output', ASSET_WORK_ASSET_WORK_ASSET_WORK_ASSET_WORK_ASSET_WORK_ASSET_WORK_ROOT / 'data/teacher_cache' / ('T1-full-' + suffix)]
        run(command + (['--hidden'] if SAVE_HIDDEN else []))
        run([AVA_PY, 'scripts/data/hf_assets.py', 'publish',
             'teacher_cache_t1_hidden' if SAVE_HIDDEN else 'teacher_cache_t1',
             '--directory', ASSET_WORK_ASSET_WORK_ASSET_WORK_ASSET_WORK_ASSET_WORK_ROOT / 'data/teacher_cache' / ('T1-full-' + suffix), '--dataset'])
else:
    print('선행 대기: 선택된 T1이 없습니다. 공개 T0/zero-context debug를 대신 저장하지 않습니다.')

In [ ]:
RELEASE_GPU_NOW = False  # 필요한 단계가 끝났으면 True로 바꿔 실행하십시오.
print('실제 경과 시간(시간):', round((time.time()-SESSION_STARTED_UNIX)/3600, 3))
if RELEASE_GPU_NOW:
    runtime.unassign()